# 02 — Baseline : régression logistique

Référence du KPI « coût métier −15 % » : tout modèle futur est comparé à cette baseline. Données brutes, échantillon stratifié de `params.data.sample_size` lignes, split local (l'étape `split` d'Areej n'existe pas encore). Les figures ne s'affichent pas dans le notebook (backend `Agg`) : elles sont enregistrées en PNG dans `reports/figures/`.

### Pourquoi un coût 5:1 ?

Un **défaut non détecté** (faux négatif) fait perdre une grande partie du capital prêté ; un **bon client refusé** (faux positif) ne fait perdre que la marge (les intérêts) qu'il aurait rapportée. D'où un rapport de **5 pour 1** (`cost.cost_fn = 5`, `cost.cost_fp = 1` dans `params.yaml`). Ce n'est pas une valeur du SYNOPSIS : la Roadmap le présente comme une **hypothèse de départ, à justifier**, et c'est l'argument ci-dessus qui la justifie. Une étude manuelle coûte **0,3** (temps d'analyste, D3) et le dossier étudié est supposé bien décidé. Avec un modèle calibré, le seuil unique optimal théorique est `cost_fp / (cost_fp + cost_fn) = 1/6 ≈ 0,167`.

In [ ]:
import json
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from crediscore.config import load_params
from crediscore.models.cost import ACCEPT, REFUSE, REVIEW, business_cost
from crediscore.models.threshold import optimize_thresholds

matplotlib.use("Agg")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
params = load_params(ROOT / "params.yaml")
SEED = params.data.random_state
COSTS = {"cost_fn": params.cost.cost_fn, "cost_fp": params.cost.cost_fp}

# Pending Areej's params PR (D1, D3, D8): not in params.yaml yet.
VAL_SIZE = 0.2          # data.val_size
COST_REVIEW = 0.3       # cost.cost_review
MAX_REVIEW_SHARE = 0.15 # threshold.max_review_share
MIN_RECALL = 0.65       # threshold.min_recall
COSTS["cost_review"] = COST_REVIEW

BLUE, ORANGE, GREY, INK = "#2a78d6", "#eb6834", "#8a8984", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": GREY, "axes.labelcolor": INK, "xtick.color": INK,
    "ytick.color": INK, "text.color": INK, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 9,
})


def save(fig: plt.Figure, name: str) -> None:
    fig.tight_layout()
    path = FIG_DIR / name
    fig.savefig(path, dpi=100)
    plt.close(fig)
    print(f"{path.name}: {path.stat().st_size / 1024:.0f} KB")

## 1. Features dérivées (version locale provisoire)

Les 8 features dérivées de `params.features.numeric` n'existent pas dans les données brutes. La **vraie fonction partagée `add_derived_features` sera livrée par Areej (D2)** ; celle-ci n'est qu'une copie locale pour la baseline, définie d'après les commentaires de `params.yaml` et D10.

In [ ]:
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """Local stand-in for Areej's shared function (D2), to be replaced."""
    out = df.copy()
    anom = out["DAYS_EMPLOYED"] == 365243
    ext = out[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
    out["ANNUITY_INCOME_RATIO"] = out["AMT_ANNUITY"] / out["AMT_INCOME_TOTAL"]
    out["CREDIT_INCOME_RATIO"] = out["AMT_CREDIT"] / out["AMT_INCOME_TOTAL"]
    out["CREDIT_TERM"] = out["AMT_ANNUITY"] / out["AMT_CREDIT"]
    out["AGE_YEARS"] = -out["DAYS_BIRTH"] / 365.25
    out["EMPLOYED_YEARS"] = (-out["DAYS_EMPLOYED"] / 365.25).mask(anom)
    out["DAYS_EMPLOYED_ANOM"] = anom.astype(int)
    out["EXT_SOURCE_MISSING_COUNT"] = ext.isna().sum(axis=1)
    out["IS_THIN_FILE"] = (ext["EXT_SOURCE_1"].isna()
                           & ext["EXT_SOURCE_3"].isna()).astype(int)
    return out


raw = pd.read_csv(ROOT / params.data.raw_path)
target = params.target
# Same stratified sample size as the future featurize stage (contract 1).
df, _ = train_test_split(raw, train_size=params.data.sample_size,
                         stratify=raw[target], random_state=SEED)
df = add_derived_features(df)
features = params.features.all
assert not set(features) & set(params.sensitive_cols + [params.id_col, target])
missing_cols = set(features) - set(df.columns)
assert not missing_cols, missing_cols
print(df.shape, f"default rate {df[target].mean():.4f}")

## 2. Split stratifié : train / validation / test (D1)

In [ ]:
train_full, test = train_test_split(df, test_size=params.data.test_size,
                                    stratify=df[target], random_state=SEED)
train, val = train_test_split(train_full, test_size=VAL_SIZE,
                              stratify=train_full[target], random_state=SEED)
for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s} {len(part):6d} rows, default rate {part[target].mean():.4f}")
X_train, y_train = train[features], train[target].to_numpy()
X_val, y_val = val[features], val[target].to_numpy()
X_test, y_test = test[features], test[target].to_numpy()

## 3. Pipeline et entraînement

Numériques : imputation médiane + standardisation. Catégorielles : imputation par la modalité la plus fréquente + one-hot (`handle_unknown="ignore"`). `class_weight="balanced"` (T5). La calibration isotonique se fait par CV 5 plis **sur le train** (T2), avec `ensemble=False` : le pipeline est réajusté sur tout le train et une seule fonction isotonique est apprise sur les prédictions hors-pli. Le modèle servi est donc une fonction monotone de ce pipeline, celui que SHAP explique. La validation ne sert qu'aux seuils.

In [ ]:
def build_pipeline() -> Pipeline:
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
         params.features.numeric),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OneHotEncoder(handle_unknown="ignore")),
         params.features.categorical),
    ])
    model = LogisticRegression(class_weight="balanced", max_iter=2000,
                               random_state=SEED)
    return Pipeline([("pre", pre), ("model", model)])


pipe = build_pipeline().fit(X_train, y_train)
calibrated = CalibratedClassifierCV(build_pipeline(), cv=5, method="isotonic",
                                    ensemble=False)
calibrated.fit(X_train, y_train)
# The pipeline inside the served model is the full-train one SHAP explains.
inner = calibrated.calibrated_classifiers_[0].estimator
assert len(calibrated.calibrated_classifiers_) == 1
assert np.allclose(inner[-1].coef_, pipe[-1].coef_)

proba = {
    ("raw", "val"): pipe.predict_proba(X_val)[:, 1],
    ("raw", "test"): pipe.predict_proba(X_test)[:, 1],
    ("cal", "val"): calibrated.predict_proba(X_val)[:, 1],
    ("cal", "test"): calibrated.predict_proba(X_test)[:, 1],
}
ys = {"val": y_val, "test": y_test}
rows = []
for (kind, split), p in proba.items():
    rows.append({"model": kind, "split": split,
                 "roc_auc": roc_auc_score(ys[split], p),
                 "pr_auc": average_precision_score(ys[split], p),
                 "brier": brier_score_loss(ys[split], p)})
quality = pd.DataFrame(rows).set_index(["model", "split"])
prior = y_train.mean()
print(quality.round(4).to_string())
print(f"Brier of a constant {prior:.4f}: "
      f"val {brier_score_loss(y_val, np.full_like(y_val, prior, float)):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for split, color in [("val", BLUE), ("test", ORANGE)]:
    y, p = ys[split], proba[("cal", split)]
    fpr, tpr, _ = roc_curve(y, p)
    axes[0].plot(fpr, tpr, color=color, lw=2,
                 label=f"{split} AUC {roc_auc_score(y, p):.3f}")
    prec, rec, _ = precision_recall_curve(y, p)
    axes[1].plot(rec, prec, color=color, lw=2,
                 label=f"{split} AP {average_precision_score(y, p):.3f}")
axes[0].plot([0, 1], [0, 1], color=GREY, ls="--", lw=1)
axes[0].set(xlabel="Taux de faux positifs", ylabel="Rappel (classe 1)",
            title="ROC (modèle calibré)")
axes[1].axhline(prior, color=GREY, ls="--", lw=1)
axes[1].set(xlabel="Rappel (classe 1)", ylabel="Précision",
            title="Précision-rappel (calibré)")
for kind, color, label in [("raw", ORANGE, "non calibré"), ("cal", BLUE, "calibré")]:
    frac, mean_p = calibration_curve(y_val, proba[(kind, "val")], n_bins=10,
                                     strategy="quantile")
    axes[2].plot(mean_p, frac, color=color, lw=2, marker="o", ms=4,
                 label=f"{label} (Brier {brier_score_loss(y_val, proba[(kind, 'val')]):.3f})")
axes[2].plot([0, 1], [0, 1], color=GREY, ls="--", lw=1)
axes[2].set(xlabel="Probabilité prédite", ylabel="Taux de défaut observé",
            title="Calibration (validation)")
for ax in axes:
    ax.legend(frameon=False, loc="best")
save(fig, "baseline_roc_pr_calibration.png")

## 4. Seuils de décision (sur la validation, probabilités calibrées)

Décision : `ACCEPTE` si p < low, `A_ETUDIER` si low ≤ p < high, `REFUSE` si p ≥ high (low = high : 2 zones). Recall (D8) : un défaut envoyé en étude compte comme détecté. Recherche **exacte** : les seuils candidats sont toutes les probabilités distinctes de la validation (chaque frontière de décision possible) ; le coût de chaque optimum est revérifié avec `business_cost`. **Règle du milieu** : chaque seuil retenu est ensuite placé au milieu entre son palier et le palier distinct immédiatement inférieur (le palier lui-même s'il n'y en a pas). Les décisions sur la validation sont identiques (vérifié), mais elles ne dépendent plus d'une égalité exacte de flottants : un arrondi de la réponse ou une proba en float32 ne fait plus basculer tout un palier. Même règle que `crediscore.models.threshold.optimize_thresholds` (vérifié pour D3). Les seuils sont écrits en pleine précision dans `reports/baseline_metrics.json`. `recall_margin` = recall − 0,65.

In [ ]:
def decide(p: np.ndarray, low: float, high: float) -> np.ndarray:
    return np.where(p < low, ACCEPT, np.where(p < high, REVIEW, REFUSE))


def evaluate(y: np.ndarray, p: np.ndarray, low: float, high: float) -> dict:
    d = decide(p, low, high)
    return {"low": low, "high": high,
            "cost": business_cost(y, d, **COSTS),
            "cost_per_file": business_cost(y, d, **COSTS) / len(y),
            "recall": float(np.mean(d[y == 1] != ACCEPT)),
            "recall_margin": float(np.mean(d[y == 1] != ACCEPT)) - MIN_RECALL,
            "review_share": float(np.mean(d == REVIEW)),
            "accept_share": float(np.mean(d == ACCEPT))}


def exact_search(y: np.ndarray, p: np.ndarray, *, min_recall: float = 0.0,
                 max_review_share: float = 1.0, two_zones: bool = False):
    """Cost-optimal (low, high) over every distinct cut-off, with counts."""
    cuts = np.r_[np.unique(p), 1.001]             # p < cut: every boundary
    below = np.searchsorted(np.sort(p), cuts, side="left")
    pos_below = np.searchsorted(np.sort(p[y == 1]), cuts, side="left")
    n, n_pos, n_neg = len(y), int(y.sum()), int((y == 0).sum())
    best_cost, best = np.inf, None
    for i in range(len(cuts)):
        if 1 - pos_below[i] / n_pos < min_recall:
            continue
        j = np.array([i]) if two_zones else np.arange(i, len(cuts))
        review = below[j] - below[i]
        cost = (COSTS["cost_fn"] * pos_below[i]
                + COSTS["cost_fp"] * (n_neg - (below[j] - pos_below[j]))
                + COSTS["cost_review"] * review)
        cost = np.where(review / n <= max_review_share, cost, np.inf)
        k = int(np.argmin(cost))
        if cost[k] < best_cost:
            best_cost, best = cost[k], (float(cuts[i]), float(cuts[j[k]]))
    # Rule A: move each cut to the midpoint below its step (same decisions on
    # this data, robust to rounding and float32). The 1.001 sentinel is kept.
    steps = np.unique(p)

    def midpoint(c: float) -> float:
        lower = steps[steps < c]
        return c if c > 1 or lower.size == 0 else float((c + lower[-1]) / 2)

    low, high = midpoint(best[0]), midpoint(best[1])
    assert np.array_equal(decide(p, *best), decide(p, low, high))
    res = evaluate(y, p, low, high)
    assert np.isclose(res["cost"], best_cost)   # cross-check with cost.py
    return res


p_val, p_test = proba[("cal", "val")], proba[("cal", "test")]
print("distinct calibrated probabilities on val:", len(np.unique(p_val)))
policies = {
    "accept_all": evaluate(y_val, p_val, 1.001, 1.001),
    "single_threshold": exact_search(y_val, p_val, two_zones=True),
    "single_threshold_recall65": exact_search(y_val, p_val, two_zones=True,
                                              min_recall=MIN_RECALL),
    "three_zones_unconstrained": exact_search(y_val, p_val),
    "three_zones_D3": exact_search(y_val, p_val, min_recall=MIN_RECALL,
                                   max_review_share=MAX_REVIEW_SHARE),
}
# Same optimum as the package function the training stage uses.
d3 = policies["three_zones_D3"]
assert (d3["low"], d3["high"]) == optimize_thresholds(
    y_val, p_val, **COSTS, min_recall=MIN_RECALL, max_review_share=MAX_REVIEW_SHARE)
val_table = pd.DataFrame(policies).T
print(val_table.round(4).to_string())

## 5. Application des seuils choisis sur le test

In [ ]:
test_table = pd.DataFrame({
    name: evaluate(y_test, p_test, pol["low"], pol["high"])
    for name, pol in policies.items()
}).T
print(test_table.round(4).to_string())

grid = np.linspace(0.01, 0.6, 120)
fig, ax = plt.subplots(figsize=(6, 3.4))
for split, y, p, color in [("val", y_val, p_val, BLUE), ("test", y_test, p_test, ORANGE)]:
    per_file = [evaluate(y, p, t, t)["cost_per_file"] for t in grid]
    ax.plot(grid, per_file, color=color, lw=2, label=split)
best = policies["single_threshold"]["low"]
ax.axvline(best, color=GREY, ls="--", lw=1)
ax.text(best, ax.get_ylim()[1], f" seuil optimal {best:.3f}", va="top", color=INK)
ax.set(xlabel="Seuil unique sur la probabilité calibrée",
       ylabel="Coût moyen par dossier", title="Coût métier selon le seuil (2 zones)")
ax.legend(frameon=False)
save(fig, "baseline_cost_vs_threshold.png")

## 6. Équité : écarts d'acceptation (D5, D8)

Sur le **test**, avec les seuils 3 zones D3 (et le seuil unique pour comparaison). « Accepté » = `ACCEPTE` seulement (D8). Écart brut = parité démographique ; égalité des chances = acceptation parmi les clients qui ont remboursé (`TARGET = 0`). `XNA` est exclu et compté. Écarts en points.

In [ ]:
age_band = pd.cut(test["AGE_YEARS"], [0, 30, 50, np.inf], right=False,
                  labels=["<30", "30-50", "≥50"])
gender = test["CODE_GENDER"]
print("XNA rows excluded from the gender audit:", int((gender == "XNA").sum()))
groups = {"gender": gender.where(gender != "XNA"), "age_band": age_band}

fair_rows, fairness = [], {}
for pol_name in ["three_zones_D3", "single_threshold"]:
    pol = policies[pol_name]
    accepted = pd.Series(decide(p_test, pol["low"], pol["high"]) == ACCEPT,
                         index=test.index)
    good = test[target] == 0
    for attr, g in groups.items():
        raw_rate = accepted.groupby(g, observed=True).mean()
        eo_rate = accepted[good].groupby(g[good], observed=True).mean()
        gaps = {"raw_gap_pts": 100 * (raw_rate.max() - raw_rate.min()),
                "equal_opportunity_gap_pts": 100 * (eo_rate.max() - eo_rate.min())}
        fairness.setdefault(pol_name, {})[attr] = {
            **{k: round(float(v), 2) for k, v in gaps.items()},
            "acceptance_by_group": raw_rate.round(4).to_dict(),
            "acceptance_good_payers_by_group": eo_rate.round(4).to_dict()}
        for grp in raw_rate.index:
            fair_rows.append({"policy": pol_name, "attribute": attr, "group": grp,
                              "n": int((g == grp).sum()),
                              "acceptance": raw_rate[grp],
                              "acceptance_good_payers": eo_rate[grp]})
        print(f"{pol_name:17s} {attr:9s} raw gap {gaps['raw_gap_pts']:5.1f} pts, "
              f"equal-opportunity gap {gaps['equal_opportunity_gap_pts']:5.1f} pts")
print(pd.DataFrame(fair_rows).round(4).to_string(index=False))

In [ ]:
fr = pd.DataFrame(fair_rows).query("policy == 'three_zones_D3'")
fig, ax = plt.subplots(figsize=(6, 3.8))
y_pos = np.arange(len(fr))
ax.barh(y_pos - 0.18, fr["acceptance"] * 100, height=0.34, color=BLUE,
        label="tous les clients (écart brut)")
ax.barh(y_pos + 0.18, fr["acceptance_good_payers"] * 100, height=0.34,
        color=ORANGE, label="bons payeurs (égalité des chances)")
ax.set_yticks(y_pos, fr["attribute"].str.replace("age_band", "âge")
              .str.replace("gender", "sexe") + " " + fr["group"].astype(str))
for yp, a, e in zip(y_pos, fr["acceptance"], fr["acceptance_good_payers"]):
    ax.text(a * 100, yp - 0.18, f" {a:.1%}", va="center")
    ax.text(e * 100, yp + 0.18, f" {e:.1%}", va="center")
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Taux d'acceptation (%) — test, seuils 3 zones D3")
ax.set_title("Acceptation par groupe", loc="left")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.4, -0.2), ncol=2)
save(fig, "baseline_fairness.png")

In [ ]:
def rounded(d: dict) -> dict:
    # Thresholds in full precision: rounding would move whole isotonic steps.
    return {k: float(v) if k in ("low", "high") else round(float(v), 4)
            for k, v in d.items()}


metrics = {
    "note": "Baseline LogReg, local split (D1), derived features computed locally "
            "until Areej's add_derived_features lands. Thresholds tuned on val.",
    "data": {"rows": len(df), "train": len(train), "val": len(val), "test": len(test),
             "seed": SEED},
    "costs": COSTS,
    "constraints": {"min_recall": MIN_RECALL, "max_review_share": MAX_REVIEW_SHARE},
    "quality": {f"{m}_{s}": rounded(v) for (m, s), v in quality.to_dict("index").items()},
    "policies_val": {k: rounded(v) for k, v in policies.items()},
    "policies_test": {k: rounded(v) for k, v in test_table.to_dict("index").items()},
    "fairness_test": fairness,
}
out = ROOT / "reports" / "baseline_metrics.json"
out.write_text(json.dumps(metrics, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(out.name, f"{out.stat().st_size / 1024:.1f} KB")

## Conclusions (baseline de référence)

Échantillon stratifié de 100 000 dossiers (seed 42) : train 64 000, validation 16 000, test 20 000. Seuils choisis sur la validation, puis appliqués tels quels au test.

**Qualité du score**

| | ROC-AUC | PR-AUC | Brier |
|---|---|---|---|
| Validation, non calibré | 0,739 | 0,224 | 0,2063 |
| Validation, calibré (isotonique, CV 5, `ensemble=False`) | 0,738 | 0,213 | **0,0689** |
| Test, non calibré | 0,746 | 0,228 | 0,2025 |
| Test, calibré | **0,746** | **0,220** | **0,0685** |

- L'AUC (0,746 sur le test) et la PR-AUC (0,220) sont **sous les cibles** du SYNOPSIS (0,75 et 0,25). C'est la référence que les modèles à arbres doivent battre.
- `class_weight="balanced"` gonfle les probabilités (Brier 0,20). Après calibration, le Brier tombe à **0,069**, juste sous la cible 0,07. Un prédicteur constant fait 0,074 (T6) : la marge est mince.
- Avec `ensemble=False`, la fonction isotonique n'a que **69 valeurs distinctes** sur la validation : beaucoup de dossiers ont exactement la même probabilité. La PR-AUC perd un peu (0,228 → 0,220 sur le test) et les seuils sont placés entre deux paliers (règle du milieu).

**Coût métier** (coût moyen par dossier : la validation et le test n'ont pas la même taille ; marge de recall = recall − 65 %)

| Politique (seuils de la validation) | low | high | Recall val / test | Marge val / test | Part en étude test | Coût/dossier val | Coût/dossier test |
|---|---|---|---|---|---|---|---|
| Tout accepter | – | – | 0 % | – | 0 % | 0,404 | 0,404 |
| Seuil unique optimal | 0,1666 | 0,1666 | 37,5 % / 38,6 % | – | 0 % | 0,348 | **0,340** |
| Seuil unique avec recall ≥ 65 % | 0,0898 | 0,0898 | 66,2 % / 66,5 % | +1,2 / +1,5 pt | 0 % | 0,418 | 0,409 |
| 3 zones sans contrainte | 0,0562 | 0,6250 | 79,9 % / 79,4 % | – | 48,7 % | 0,230 | 0,229 |
| **3 zones D3** (recall ≥ 65 %, étude ≤ 15 %) | **0,0898** | **0,1232** | **66,2 % / 66,5 %** | **+1,2 / +1,5 pt** | **14,2 %** | 0,333 | **0,325** |

- Recherche exacte sur les 69 probabilités distinctes de la validation : c'est un vrai optimum, pas l'approximation d'une grille. Les seuils du tableau sont les milieux entre paliers, arrondis pour la lecture ; le JSON les garde en pleine précision.
- Le seuil unique optimal (palier 0,171, coupure au milieu 0,167) retrouve la théorie (1/6 ≈ 0,167), mais son recall n'est que de **38 %**.
- Imposer recall ≥ 65 % avec un seul seuil coûte **plus cher que tout accepter**. Les 3 zones résolvent ce conflit : recall 66,5 % et coût 0,325 par dossier sur le test, soit 4,4 % de moins que le seuil unique.
- Sans plafond, l'optimum envoie **49 % des dossiers en étude**. Cela confirme D3 : le plafond de 15 % est indispensable. La contrainte de recall est saturée (marge de 1,2 pt seulement sur la validation). Le plafond n'est pas atteint exactement (14,2 %) à cause des paliers de la fonction isotonique.
- **Référence du KPI −15 %** : coût de la baseline 3 zones D3 sur ce test = **6 495,6** (0,325 par dossier). La comparaison n'est valable que **sur le même jeu de test**. Au lot 3, `evaluate` **recalcule cette baseline LogReg sur le split de test d'Areej**, avec les mêmes contraintes, et compare le nouveau modèle à ce recalcul, pas au chiffre ci-dessus.

**Équité (test, 3 zones D3 ; « accepté » = `ACCEPTE` seulement, D8)**

| Attribut | Écart brut | Égalité des chances (bons payeurs) |
|---|---|---|
| Sexe (F 69,9 %, M 62,1 %) | 7,8 pts | 6,6 pts |
| Âge (<30 : 44,3 %, 30-50 : 66,0 %, ≥50 : 78,8 %) | **34,5 pts** | **32,9 pts** |

Avec le seuil unique (plus permissif) : sexe 5,9 / 4,5 pts, âge 22,2 / 19,0 pts. Aucune ligne `XNA` dans le test (4 dans toute la base).

**Mesure pour D5 :** l'écart brut dépasse 10 pts pour l'âge (34,5 pts), pas pour le sexe (7,8 pts). L'égalité des chances ne ramène pas l'écart d'âge sous 10 pts (32,9 pts parmi les bons payeurs). **D5 tranchée le 25/10/2026** : seuil absolu de 10 pts sur le **sexe** (brut et égalité des chances ; la baseline passe avec 7,8 / 6,6 pts) ; pour l'**âge**, non-régression (égalité des chances du challenger ≤ champion + 2 pts), avec les deux écarts publiés à chaque évaluation.
